# Интерактивный селектор стали и соединений по СП 16.13330.2017

Настоящий блокнот предназначен для автоматизированного выбора и анализа расчетных сопротивлений прокатной стали, болтовых соединений и деформационных диаграмм в соответствии с требованиями **СП 16.13330.2017** *«Стальные конструкции»* (с Изменениями № 1 и № 2).

### Реализованные нормативные требования:
1. **Точная градация по толщинам** ($t$): для фасонного проката толщина полки $t_f$ (*табл. В.5*), для листового проката и труб (*табл. В.3*), для прокатных двутавров с параллельными гранями полок по ГОСТ Р 57837 (*табл. В.4*).
2. **Учет статистического контроля свойств** (*сноски к табл. В.3, В.5*): числитель ($\gamma_m = 1{,}025$) или знаменатель ($\gamma_m = 1{,}050$).
3. **Полный набор расчетных сопротивлений**: растяжение/сжатие/изгиб ($R_y$), сопротивление по пределу прочности ($R_u$), сдвиг ($R_s = 0{,}58 R_y$), смятие торцевой поверхности ($R_p$), смятие в шарнирах ($R_{lp}$).
4. **Коэффициенты условий работы** $\gamma_c$ (*Таблица 1*).
5. **Болтовые соединения**: расчетные сопротивления срезу ($R_{bs}$), растяжению ($R_{bt}$) и несущая способность болтов по *таблице Г.5*.

In [1]:
import sys
from pathlib import Path

if '.' not in sys.path:
    sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output

from sp16_materials import (
    StructuralSteel, SteelBolt,
    list_steel_grades, generate_steel_code_snippet
)

print("Модуль sp16_materials успешно загружен.")

Модуль sp16_materials успешно загружен.


## 1. Интерактивная панель выбора стали и соединений

In [2]:
# Виджеты управления
w_profile_type = widgets.Dropdown(
    options=[
        ('Фасонный прокат (ГОСТ 27772, табл. В.5)', 'shapes'),
        ('Листовой и универсальный прокат (табл. В.3)', 'plates'),
        ('Двутавры с параллельными гранями полок (ГОСТ Р 57837, табл. В.4)', 'beams_parallel'),
        ('Трубы круглые и профильные (табл. В.3)', 'tubes')
    ],
    value='shapes',
    description='Вид проката:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='95%')
)

w_steel_grade = widgets.Dropdown(
    options=list_steel_grades('shapes'),
    value='С255',
    description='Марка стали:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='95%')
)

w_thickness = widgets.FloatSlider(
    value=12.0,
    min=2.0,
    max=80.0,
    step=1.0,
    description='Толщина t (tf), мм:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='95%')
)

w_stat_control = widgets.Checkbox(
    value=True,
    description='Статистический контроль свойств (ГОСТ 27772, числитель / gamma_m=1.025)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='98%')
)

w_gamma_c = widgets.Dropdown(
    options=[
        ('Рядовые элементы конструкций (gamma_c = 1.00)', 1.00),
        ('Колонны жилых и общественных зданий (gamma_c = 0.95)', 0.95),
        ('Балки под трибунами, залами, архивами (gamma_c = 0.90)', 0.90),
        ('Особые условия / стесненные деформации (gamma_c = 0.85)', 0.85)
    ],
    value=1.00,
    description='Условия работы:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='95%')
)

# Болты
w_bolt_grade = widgets.Dropdown(
    options=['5.6', '5.8', '8.8', '10.9', '12.9'],
    value='8.8',
    description='Класс болта:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='95%')
)

w_bolt_diameter = widgets.Dropdown(
    options=[12, 16, 20, 24, 27, 30, 36],
    value=20,
    description='Диаметр резьбы:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='95%')
)

out_panel = widgets.Output()

def on_profile_change(change):
    new_grades = list_steel_grades(w_profile_type.value)
    w_steel_grade.options = new_grades
    if 'С255' in new_grades:
        w_steel_grade.value = 'С255'
    elif 'С255Б' in new_grades:
        w_steel_grade.value = 'С255Б'
    else:
        w_steel_grade.value = new_grades[0]

w_profile_type.observe(on_profile_change, names='value')

def update_view(*args):
    with out_panel:
        clear_output(wait=True)
        
        try:
            steel = StructuralSteel(
                grade=w_steel_grade.value,
                profile_type=w_profile_type.value,
                thickness=w_thickness.value,
                statistical_control=w_stat_control.value,
                gamma_c=w_gamma_c.value
            )
        except ValueError as e:
            print(f"Предупреждение: {e}")
            return
            
        bolt = SteelBolt(
            grade=w_bolt_grade.value,
            diameter=w_bolt_diameter.value,
            gamma_b=1.0
        )
        
        # Графика
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8), dpi=110)
        plt.subplots_adjust(wspace=0.25)
        
        # 1. Диаграмма Прандтля стали
        eps_arr, sig_arr = steel.get_diagram(model='prandtl', n_points=120)
        ax1.plot(eps_arr * 1000, sig_arr, color='#1f77b4', lw=2.4, label=f'{steel.grade} (t={steel.thickness:.0f} мм)')
        ax1.axhline(steel.Ry, color='#d62728', linestyle='--', alpha=0.7, label=f'Ry = {steel.Ry} МПа')
        ax1.axhline(steel.Ru, color='#7f7f7f', linestyle=':', alpha=0.7, label=f'Ru = {steel.Ru} МПа')
        ax1.scatter([steel.eps_y * 1000], [steel.Ry], color='#d62728', zorder=5)
        ax1.annotate(f'Текучесть ({steel.eps_y*1000:.2f}‰; {steel.Ry} МПа)',
                     xy=(steel.eps_y * 1000, steel.Ry),
                     xytext=(steel.eps_y * 1000 + 1.5, steel.Ry * 0.75),
                     arrowprops=dict(arrowstyle='->', color='#d62728', lw=1.2),
                     fontsize=9, fontweight='bold', color='#d62728')
        ax1.set_title(f'Диаграмма деформирования стали {steel.grade}', fontsize=12, fontweight='bold', pad=10)
        ax1.set_xlabel('Относительная деформация $\\varepsilon$, ‰', fontsize=10)
        ax1.set_ylabel('Напряжение $\\sigma$, МПа', fontsize=10)
        ax1.grid(True, linestyle=':', alpha=0.6)
        ax1.legend(loc='lower right', fontsize=9)
        
        # 2. Несущая способность болта М по диаметрам
        d_list = [12, 16, 20, 24, 27, 30, 36]
        nbs_list = [SteelBolt(w_bolt_grade.value, d).shear_capacity(1) for d in d_list]
        nbt_list = [SteelBolt(w_bolt_grade.value, d).tension_capacity() or 0 for d in d_list]
        
        x = np.arange(len(d_list))
        width = 0.38
        ax2.bar(x - width/2, nbs_list, width, label='Срез Nbs (1 пл.)', color='#ff7f0e', alpha=0.85)
        ax2.bar(x + width/2, nbt_list, width, label='Растяжение Nbt', color='#2ca02c', alpha=0.85)
        ax2.set_xticks(x)
        ax2.set_xticklabels([f'М{d}' for d in d_list])
        ax2.set_title(f'Несущая способность болтов кл. {w_bolt_grade.value}, кН', fontsize=12, fontweight='bold', pad=10)
        ax2.set_xlabel('Диаметр болта', fontsize=10)
        ax2.set_ylabel('Усилие, кН', fontsize=10)
        ax2.grid(True, linestyle=':', alpha=0.6, axis='y')
        ax2.legend(loc='upper left', fontsize=9)
        
        plt.show()
        
        # Вывод Markdown
        display(Markdown(steel.to_markdown()))
        display(Markdown(bolt.to_markdown()))
        
        # Сниппет
        snippet = generate_steel_code_snippet(steel, bolt)
        display(Markdown("### Готовый фрагмент кода для вставки в расчетный блокнот:"))
        display(Markdown(f"```python\n{snippet}\n```"))

for w in [w_profile_type, w_steel_grade, w_thickness, w_stat_control, w_gamma_c, w_bolt_grade, w_bolt_diameter]:
    w.observe(update_view, names='value')

# Разметка
box_steel = widgets.VBox([
    widgets.HTML("<b>Параметры прокатной стали:</b>"),
    w_profile_type, w_steel_grade, w_thickness, w_stat_control, w_gamma_c
], layout=widgets.Layout(padding='10px', border='1px solid #ddd', margin='4px', border_radius='6px'))

box_bolts = widgets.VBox([
    widgets.HTML("<b>Параметры болтовых соединений:</b>"),
    w_bolt_grade, w_bolt_diameter
], layout=widgets.Layout(padding='10px', border='1px solid #ddd', margin='4px', border_radius='6px'))

ui = widgets.HBox([box_steel, box_bolts], layout=widgets.Layout(width='100%'))
display(ui)
display(out_panel)


Output()

## 2. Пример расчетного использования в инженерных задачах

Расчет несущей способности изгибаемой балки с проверкой фланцевого болтового соединения:

In [3]:
# Пример расчета несущей способности балки из стали С255 по моменту (п. 8.2 СП 16)
steel_beam = StructuralSteel('С255', profile_type='shapes', thickness=14.0, gamma_c=1.0)

# Геометрические характеристики двутавра 30Б1 (СТО АСЧМ 20-93):
# Wx = 472.4 см3 = 472.4e3 мм3
Wx = 472.4e3  # мм3

# 1. Предельный упругий изгибающий момент:
M_el = Wx * steel_beam.Ry / 1e6  # кН*м
print(f"Сталь {steel_beam.grade} (t={steel_beam.thickness} мм): Ry = {steel_beam.Ry} МПа")
print(f"Упругий предельный момент: M_el = {M_el:.2f} кН*м")

# 2. Проверка опорного узла на 4 болтах М20 кл. 8.8 на сдвиг опорной реакции Q = 120 кН:
bolt_conn = SteelBolt('8.8', diameter=20, gamma_b=0.9)
n_bolts = 4
Q_max = n_bolts * bolt_conn.shear_capacity(n_shear_planes=1)
print(f"Несущая способность болтового узла из 4xМ20 кл. 8.8: Q_max = {Q_max:.1f} кН (запас: {Q_max / 120.0:.2f})")

Сталь С255 (t=14.0 мм): Ry = 240.0 МПа
Упругий предельный момент: M_el = 113.38 кН*м
Несущая способность болтового узла из 4xМ20 кл. 8.8: Q_max = 375.5 кН (запас: 3.13)
